# ▶️ Init data 

## directory set-up 

In [ ]:
%load_ext autoreload
%autoreload 2

## check the roots

In [ ]:
import sys
from pathlib import Path

# This finds the directory of the current notebook and goes up to the project root
# Adjust the number of '.parent' calls depending on how deep your notebook is
root_dir = Path.cwd() 

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

print(f"Project root added to path: {root_dir}")

## packages set-up

In [ ]:
import os # manage directories

# python librairies
import time
import numpy as np
import pandas as pd
import pickle # saving and loading file using pikle

# for graphical visualisation
# !pip install ipympl # better
%matplotlib ipympl 
# less installation required
# %matplotlib inline 
# !pip install seaborn
import seaborn as sns
import matplotlib.ticker as ticker
import matplotlib.pyplot as plt
#!pip install mplcursors
import mplcursors

#brightway librairies
import bw2io as bi 
import bw2data as bd
import bw2calc as bc

# local modules: circularity and lca
from circularity_lci.burden_free_analyzer import BurdenFreeAnalyzer
from circularity_lci.biosphere_flow_manager import BiosphereFlowManager
from circularity_lci.progress_tracker import ProgressTracker
from circularity_lci.circularity_calculator import CircularityCalculator
from circularity_lci.multi_lca_calculator import MultiLCACalculator
from circularity_lci.circularity_database_analyzer import CircularityDatabaseAnalyzer


In [ ]:
import sys
sys.path.append(r"C:\Users\lfreboeuf\github\circularity_lci\src")
import circularitylci as clci

## database (ecoinvent) set-up

### credentials info

Before, in **your** local python environment, update your credentials with the commands:  
`conda env config vars set EI_USERNAME=ton_id_ecoinvent`  
`conda env config vars set EI_PASSWORD=ton_mot_de_passe`


In [ ]:
# Récupère les identifiants depuis les variables d'environnement
ei_username = os.getenv("EI_USERNAME")
ei_password = os.getenv("EI_PASSWORD")

### unit process classification

In [ ]:

# Mapping Section -> Full name
isic_section_map = {
    "A": "Agriculture, forestry and fishing",
    "B": "Mining and quarrying",
    "C": "Manufacturing",
    "D": "Electricity, gas, steam and air conditioning supply",
    "E": "Water supply; sewerage, waste management and remediation",
    "F": "Construction",
    "G": "Wholesale and retail trade; repair of motor vehicles",
    "H": "Transportation and storage",
    "I": "Accommodation and food service activities",
    "J": "Information and communication",
    "K": "Financial and insurance activities",
    "L": "Real estate activities",
    "M": "Professional, scientific and technical activities",
    "N": "Administrative and support service activities",
    "O": "Public administration and defence; compulsory social security",
    "P": "Education",
    "Q": "Human health and social work activities",
    "R": "Arts, entertainment and recreation",
    "S": "Other service activities",
    "T": "Activities of households as employers",
    "U": "Activities of extraterritorial organizations"
}
# Mapping Division
isic_division_map = {
    # SECTION A: Agriculture, forestry and fishing
    "01": ("A", "Crop and animal production, hunting and related service activities"),
    "02": ("A", "Forestry and logging"),
    "03": ("A", "Fishing and aquaculture"),

    # SECTION B: Mining and quarrying
    "05": ("B", "Mining of coal and lignite"),
    "06": ("B", "Extraction of crude petroleum and natural gas"),
    "07": ("B", "Mining of metal ores"),
    "08": ("B", "Other mining and quarrying"),
    "09": ("B", "Mining support service activities"),

    # SECTION C: Manufacturing
    "10": ("C", "Manufacture of food products"),
    "11": ("C", "Manufacture of beverages"),
    "12": ("C", "Manufacture of tobacco products"),
    "13": ("C", "Manufacture of textiles"),
    "14": ("C", "Manufacture of wearing apparel"),
    "15": ("C", "Manufacture of leather and related products"),
    "16": ("C", "Manufacture of wood and of products of wood and cork, except furniture; manufacture of articles of straw and plaiting materials"),
    "17": ("C", "Manufacture of paper and paper products"),
    "18": ("C", "Printing and reproduction of recorded media"),
    "19": ("C", "Manufacture of coke and refined petroleum products"),
    "20": ("C", "Manufacture of chemicals and chemical products"),
    "21": ("C", "Manufacture of basic pharmaceutical products and pharmaceutical preparations"),
    "22": ("C", "Manufacture of rubber and plastics products"),
    "23": ("C", "Manufacture of other non-metallic mineral products"),
    "24": ("C", "Manufacture of basic metals"),
    "25": ("C", "Manufacture of fabricated metal products, except machinery and equipment"),
    "26": ("C", "Manufacture of computer, electronic and optical products"),
    "27": ("C", "Manufacture of electrical equipment"),
    "28": ("C", "Manufacture of machinery and equipment n.e.c."),
    "29": ("C", "Manufacture of motor vehicles, trailers and semi-trailers"),
    "30": ("C", "Manufacture of other transport equipment"),
    "31": ("C", "Manufacture of furniture"),
    "32": ("C", "Other manufacturing"),
    "33": ("C", "Repair and installation of machinery and equipment"),

    # SECTION D: Electricity, gas, steam and air conditioning supply
    "35": ("D", "Electricity, gas, steam and air conditioning supply"),

    # SECTION E: Water supply; sewerage, waste management and remediation activities
    "36": ("E", "Water collection, treatment and supply"),
    "37": ("E", "Sewerage"),
    "38": ("E", "Waste collection, treatment and disposal activities; materials recovery"),
    "39": ("E", "Remediation activities and other waste management services"),

    # SECTION F: Construction
    "41": ("F", "Construction of buildings"),
    "42": ("F", "Civil engineering"),
    "43": ("F", "Specialized construction activities"),

    # SECTION G: Wholesale and retail trade; repair of motor vehicles and motorcycles
    "45": ("G", "Wholesale and retail trade and repair of motor vehicles and motorcycles"),
    "46": ("G", "Wholesale trade, except of motor vehicles and motorcycles"),
    "47": ("G", "Retail trade, except of motor vehicles and motorcycles"),

    # SECTION H: Transportation and storage
    "49": ("H", "Land transport and transport via pipelines"),
    "50": ("H", "Water transport"),
    "51": ("H", "Air transport"),
    "52": ("H", "Warehousing and support activities for transportation"),
    "53": ("H", "Postal and courier activities"),

    # SECTION I: Accommodation and food service activities
    "55": ("I", "Accommodation"),
    "56": ("I", "Food and beverage service activities"),

    # SECTION J: Information and communication
    "58": ("J", "Publishing activities"),
    "59": ("J", "Motion picture, video and television programme production, sound recording and music publishing activities"),
    "60": ("J", "Programming and broadcasting activities"),
    "61": ("J", "Telecommunications"),
    "62": ("J", "Computer programming, consultancy and related activities"),
    "63": ("J", "Information service activities"),

    # SECTION K: Financial and insurance activities
    "64": ("K", "Financial service activities, except insurance and pension funding"),
    "65": ("K", "Insurance, reinsurance and pension funding, except compulsory social security"),
    "66": ("K", "Activities auxiliary to financial service and insurance activities"),

    # SECTION L: Real estate activities
    "68": ("L", "Real estate activities"),

    # SECTION M: Professional, scientific and technical activities
    "69": ("M", "Legal and accounting activities"),
    "70": ("M", "Activities of head offices; management consultancy activities"),
    "71": ("M", "Architectural and engineering activities; technical testing and analysis"),
    "72": ("M", "Scientific research and development"),
    "73": ("M", "Advertising and market research"),
    "74": ("M", "Other professional, scientific and technical activities"),
    "75": ("M", "Veterinary activities"),

    # SECTION N: Administrative and support service activities
    "77": ("N", "Rental and leasing activities"),
    "78": ("N", "Employment activities"),
    "79": ("N", "Travel agency, tour operator, reservation service and related activities"),
    "80": ("N", "Security and investigation activities"),
    "81": ("N", "Services to buildings and landscape activities"),
    "82": ("N", "Office administrative, office support and other business support activities"),

    # SECTION O: Public administration and defence; compulsory social security
    "84": ("O", "Public administration and defence; compulsory social security"),

    # SECTION P: Education
    "85": ("P", "Education"),

    # SECTION Q: Human health and social work activities
    "86": ("Q", "Human health activities"),
    "87": ("Q", "Residential care activities"),
    "88": ("Q", "Social work activities without accommodation"),

    # SECTION R: Arts, entertainment and recreation
    "90": ("R", "Creative, arts and entertainment activities"),
    "91": ("R", "Libraries, archives, museums and other cultural activities"),
    "92": ("R", "Gambling and betting activities"),
    "93": ("R", "Sports activities and amusement and recreation activities"),

    # SECTION S: Other service activities
    "94": ("S", "Activities of membership organizations"),
    "95": ("S", "Repair of computers and personal and household goods"),
    "96": ("S", "Other personal service activities"),

    # SECTION T: Activities of households as employers; undifferentiated goods- and services-producing activities of households for own use
    "97": ("T", "Activities of households as employers of domestic personnel"),
    "98": ("T", "Undifferentiated goods- and services-producing activities of private households for own use"),

    # SECTION U: Activities of extraterritorial organizations and bodies
    "99": ("U", "Activities of extraterritorial organizations and bodies"),
}

# Dictionary: from ecoinvent location to UN regions
location_to_un_group = {
    'RME': 'Asia-Pacific',
    'TN': 'Africa',
    'BG': 'Eastern Europe',
    'UN-EASIA': 'Asia-Pacific',
    'GI': 'WEOG',
    'RS': 'Eastern Europe',
    'IN-LD': 'Asia-Pacific',
    'US-AL': 'WEOG',
    'TR': 'WEOG',
    'US-RFC': 'WEOG',
    'AE': 'Asia-Pacific',
    'AL': 'Eastern Europe',
    'AM': 'Asia-Pacific',
    'AO': 'Africa',
    'AR': 'GRULAC',
    'AT': 'WEOG',
    'AU': 'WEOG',
    'AU-NSW': 'WEOG',
    'AU-QLD': 'WEOG',
    'AU-SA': 'WEOG',
    'AU-TAS': 'WEOG',
    'AU-VIC': 'WEOG',
    'AU-WA': 'WEOG',
    'AZ': 'Asia-Pacific',
    'BA': 'Eastern Europe',
    'BD': 'Asia-Pacific',
    'BE': 'WEOG',
    'BH': 'Asia-Pacific',
    'BJ': 'Africa',
    'BN': 'Asia-Pacific',
    'BO': 'GRULAC',
    'BR': 'GRULAC',
    'BR-AC': 'GRULAC',
    'BR-AL': 'GRULAC',
    'BR-AM': 'GRULAC',
    'BR-AP': 'GRULAC',
    'BR-BA': 'GRULAC',
    'BR-CE': 'GRULAC',
    'BR-DF': 'GRULAC',
    'BR-ES': 'GRULAC',
    'BR-GO': 'GRULAC',
    'BR-MA': 'GRULAC',
    'BR-MG': 'GRULAC',
    'BR-MS': 'GRULAC',
    'BR-MT': 'GRULAC',
    'BR-North-eastern grid': 'GRULAC',
    'BR-Northern grid': 'GRULAC',
    'BR-PA': 'GRULAC',
    'BR-PB': 'GRULAC',
    'BR-PE': 'GRULAC',
    'BR-PI': 'GRULAC',
    'BR-PR': 'GRULAC',
    'BR-RJ': 'GRULAC',
    'BR-RN': 'GRULAC',
    'BR-RO': 'GRULAC',
    'BR-RR': 'GRULAC',
    'BR-RS': 'GRULAC',
    'BR-SC': 'GRULAC',
    'BR-SE': 'GRULAC',
    'BR-SP': 'GRULAC',
    'BR-South-eastern/Mid-western grid': 'GRULAC',
    'BR-Southern grid': 'GRULAC',
    'BR-TO': 'GRULAC',
    'BW': 'Africa',
    'BY': 'Eastern Europe',
    'CA': 'WEOG',
    'CA-AB': 'WEOG',
    'CA-BC': 'WEOG',
    'CA-MB': 'WEOG',
    'CA-NB': 'WEOG',
    'CA-NF': 'WEOG',
    'CA-NS': 'WEOG',
    'CA-NT': 'WEOG',
    'CA-NU': 'WEOG',
    'CA-ON': 'WEOG',
    'CA-PE': 'WEOG',
    'CA-QC': 'WEOG',
    'CA-SK': 'WEOG',
    'CA-YK': 'WEOG',
    'CD': 'Africa',
    'CG': 'Africa',
    'CH': 'WEOG',
    'CI': 'Africa',
    'CL': 'GRULAC',
    'CM': 'Africa',
    'CN': 'Asia-Pacific',
    'CN-AH': 'Asia-Pacific',
    'CN-BJ': 'Asia-Pacific',
    'CN-CCG': 'Asia-Pacific',
    'CN-CQ': 'Asia-Pacific',
    'CN-CSG': 'Asia-Pacific',
    'CN-ECGC': 'Asia-Pacific',
    'CN-FJ': 'Asia-Pacific',
    'CN-GD': 'Asia-Pacific',
    'CN-GS': 'Asia-Pacific',
    'CN-GX': 'Asia-Pacific',
    'CN-GZ': 'Asia-Pacific',
    'CN-HA': 'Asia-Pacific',
    'CN-HB': 'Asia-Pacific',
    'CN-HE': 'Asia-Pacific',
    'CN-HL': 'Asia-Pacific',
    'CN-HN': 'Asia-Pacific',
    'CN-HU': 'Asia-Pacific',
    'CN-JL': 'Asia-Pacific',
    'CN-JS': 'Asia-Pacific',
    'CN-JX': 'Asia-Pacific',
    'CN-LN': 'Asia-Pacific',
    'CN-NCGC': 'Asia-Pacific',
    'CN-NECG': 'Asia-Pacific',
    'CN-NM': 'Asia-Pacific',
    'CN-NWG': 'Asia-Pacific',
    'CN-NX': 'Asia-Pacific',
    'CN-QH': 'Asia-Pacific',
    'CN-SA': 'Asia-Pacific',
    'CN-SC': 'Asia-Pacific',
    'CN-SD': 'Asia-Pacific',
    'CN-SGCC': 'Asia-Pacific',
    'CN-SH': 'Asia-Pacific',
    'CN-SWG': 'Asia-Pacific',
    'CN-SX': 'Asia-Pacific',
    'CN-TJ': 'Asia-Pacific',
    'CN-XJ': 'Asia-Pacific',
    'CN-XZ': 'Asia-Pacific',
    'CN-YN': 'Asia-Pacific',
    'CN-ZJ': 'Asia-Pacific',
    'CO': 'GRULAC',
    'CR': 'GRULAC',
    'CU': 'GRULAC',
    'CW': 'GRULAC',
    'CY': 'WEOG',
    'CZ': 'Eastern Europe',
    'Canada without Quebec': 'WEOG',
    'DE': 'WEOG',
    'DK': 'WEOG',
    'DO': 'GRULAC',
    'DZ': 'Africa',
    'EC': 'GRULAC',
    'EE': 'Eastern Europe',
    'EG': 'Africa',
    'ENTSO-E': 'WEOG',
    'ER': 'Africa',
    'ES': 'WEOG',
    'ET': 'Africa',
    'Europe without Austria': 'WEOG',
    'Europe without Switzerland': 'WEOG',
    'Europe without Switzerland and Austria': 'WEOG',
    'FI': 'WEOG',
    'FR': 'WEOG',
    'GA': 'Africa',
    'GB': 'WEOG',
    'GE': 'Asia-Pacific',
    'GH': 'Africa',
    'GR': 'WEOG',
    'GT': 'GRULAC',
    'HK': 'Asia-Pacific',
    'HN': 'GRULAC',
    'HR': 'Eastern Europe',
    'HT': 'GRULAC',
    'HU': 'Eastern Europe',
    'IAI Area, Africa': 'Africa',
    'IAI Area, Asia, without China and GCC': 'Asia-Pacific',
    'IAI Area, EU27 & EFTA': 'WEOG',
    'IAI Area, Gulf Cooperation Council': 'Asia-Pacific',
    'IAI Area, North America': 'WEOG',
    'IAI Area, Russia & RER w/o EU27 & EFTA': 'Eastern Europe',
    'IAI Area, South America': 'GRULAC',
    'ID': 'Asia-Pacific',
    'IE': 'WEOG',
    'IL': 'Asia-Pacific',
    'IN': 'Asia-Pacific',
    'IN-AP': 'Asia-Pacific',
    'IN-AR': 'Asia-Pacific',
    'IN-AS': 'Asia-Pacific',
    'IN-BR': 'Asia-Pacific',
    'IN-CH': 'Asia-Pacific',
    'IN-CT': 'Asia-Pacific',
    'IN-DD': 'Asia-Pacific',
    'IN-DL': 'Asia-Pacific',
    'IN-DN': 'Asia-Pacific',
    'IN-Eastern grid': 'Asia-Pacific',
    'IN-GA': 'Asia-Pacific',
    'IN-GJ': 'Asia-Pacific',
    'IN-HP': 'Asia-Pacific',
    'IN-HR': 'Asia-Pacific',
    'IN-JH': 'Asia-Pacific',
    'IN-JK': 'Asia-Pacific',
    'IN-KA': 'Asia-Pacific',
    'IN-KL': 'Asia-Pacific',
    'IN-MH': 'Asia-Pacific',
    'IN-ML': 'Asia-Pacific',
    'IN-MN': 'Asia-Pacific',
    'IN-MP': 'Asia-Pacific',
    'IN-MZ': 'Asia-Pacific',
    'IN-NL': 'Asia-Pacific',
    'IN-North-eastern grid': 'Asia-Pacific',
    'IN-Northern grid': 'Asia-Pacific',
    'IN-OR': 'Asia-Pacific',
    'IN-PB': 'Asia-Pacific',
    'IN-PY': 'Asia-Pacific',
    'IN-RJ': 'Asia-Pacific',
    'IN-SK': 'Asia-Pacific',
    'IN-Southern grid': 'Asia-Pacific',
    'IN-TN': 'Asia-Pacific',
    'IN-TR': 'Asia-Pacific',
    'IN-UP': 'Asia-Pacific',
    'IN-UT': 'Asia-Pacific',
    'IN-WB': 'Asia-Pacific',
    'IN-Western grid': 'Asia-Pacific',
    'IQ': 'Asia-Pacific',
    'IR': 'Asia-Pacific',
    'IS': 'WEOG',
    'IT': 'WEOG',
    'JM': 'GRULAC',
    'JO': 'Asia-Pacific',
    'JP': 'Asia-Pacific',
    'KE': 'Africa',
    'KG': 'Asia-Pacific',
    'KH': 'Asia-Pacific',
    'KP': 'Asia-Pacific',
    'KR': 'Asia-Pacific',
    'KW': 'Asia-Pacific',
    'KZ': 'Asia-Pacific',
    'LB': 'Asia-Pacific',
    'LK': 'Asia-Pacific',
    'LT': 'Eastern Europe',
    'LU': 'WEOG',
    'LV': 'Eastern Europe',
    'LY': 'Africa',
    'MA': 'Africa',
    'MD': 'Eastern Europe',
    'ME': 'Eastern Europe',
    'MG': 'Africa',
    'MK': 'Eastern Europe',
    'MM': 'Asia-Pacific',
    'MN': 'Asia-Pacific',
    'MT': 'WEOG',
    'MU': 'Africa',
    'MX': 'GRULAC',
    'MY': 'Asia-Pacific',
    'MZ': 'Africa',
    'NA': 'Africa',
    'NE': 'Africa',
    'NG': 'Africa',
    'NI': 'GRULAC',
    'NL': 'WEOG',
    'NO': 'WEOG',
    'NORDEL': 'WEOG',
    'NP': 'Asia-Pacific',
    'NZ': 'WEOG',
    'North America without Quebec': 'WEOG',
    'OM': 'Asia-Pacific',
    'PA': 'GRULAC',
    'PE': 'GRULAC',
    'PG': 'Asia-Pacific',
    'PH': 'Asia-Pacific',
    'PK': 'Asia-Pacific',
    'PL': 'Eastern Europe',
    'PT': 'WEOG',
    'PY': 'GRULAC',
    'QA': 'Asia-Pacific',
    'RAF': 'Africa',
    'RAS': 'Asia-Pacific',
    'RER': 'Eastern Europe',
    'RER w/o CH+DE': 'Eastern Europe',
    'RER w/o DE+NL+RU': 'Eastern Europe',
    'RER w/o RU': 'Eastern Europe',
    'RLA': 'GRULAC',
    'RNA': 'GRULAC',
    'RO': 'Eastern Europe',
    'RS': 'Eastern Europe',
    'RU': 'Eastern Europe',
    'RW': 'Africa',
    'RoE': 'WEOG',
    'RoW': 'Rest of World',
    'Russia (Asia)': 'Asia-Pacific',
    'SA': 'Asia-Pacific',
    'SAS': 'Asia-Pacific',
    'SD': 'Africa',
    'SE': 'WEOG',
    'SG': 'Asia-Pacific',
    'SI': 'Eastern Europe',
    'SK': 'Eastern Europe',
    'SN': 'Africa',
    'SS': 'Africa',
    'SV': 'GRULAC',
    'SY': 'Asia-Pacific',
    'TG': 'Africa',
    'TH': 'Asia-Pacific',
    'TJ': 'Asia-Pacific',
    'TM': 'Asia-Pacific',
    'TN': 'Africa',
    'TR': 'WEOG',
    'TT': 'GRULAC',
    'TW': 'Asia-Pacific',
    'TZ': 'Africa',
    'UA': 'Eastern Europe',
    'UCTE': 'WEOG',
    'UCTE without Germany': 'WEOG',
    'UN-EASIA': 'Asia-Pacific',
    'UN-OCEANIA': 'Asia-Pacific',
    'UN-SEASIA': 'Asia-Pacific',
    'US': 'WEOG',
    'US-ASCC': 'WEOG',
    'US-CA': 'WEOG',
    'US-CO': 'WEOG',
    'US-FL': 'WEOG',
    'US-HICC': 'WEOG',
    'US-IA': 'WEOG',
    'US-ID': 'WEOG',
    'US-IL': 'WEOG',
    'US-IN': 'WEOG',
    'US-LA': 'WEOG',
    'US-MN': 'WEOG',
    'US-MRO': 'WEOG',
    'US-ND': 'WEOG',
    'US-NE': 'WEOG',
    'US-NPCC': 'WEOG',
    'US-OH': 'WEOG',
    'US-OR': 'WEOG',
    'US-PR': 'WEOG',
    'US-RFC': 'WEOG',
    'US-SD': 'WEOG',
    'US-SERC': 'WEOG',
    'US-TRE': 'WEOG',
    'US-WA': 'WEOG',
    'US-WECC': 'WEOG',
    'US-WI': 'WEOG',
    'UY': 'GRULAC',
    'UZ': 'Asia-Pacific',
    'VE': 'GRULAC',
    'VN': 'Asia-Pacific',
    'WECC': 'WEOG',
    'WEU': 'WEOG',
    'XK': 'Eastern Europe',
    'YE': 'Asia-Pacific',
    'ZA': 'Africa',
    'ZM': 'Africa',
    'ZW': 'Africa',
    'GLO': 'Global',
} # should be in input

### flow treatment

In [ ]:
# flows to not be considered due to their time frame
excluded_flows = [
    "BOD5, Biological Oxygen Demand",
    "COD, Chemical Oxygen Demand",
    "DOC, Dissolved Organic Carbon",
    "TOC, Total Organic Carbon"
]

# valuable compartment for water flows, see function file for usability
valuable_water_compartments = ['ground-', 'surface water'] # for water flows
exclude_water = False  # Set to False if water flows are included # should be based on wet_mass or dry_mass => check dry_mass working
mass_strategy = "full_mass"      # options: "full_mass", "water_mass", "dry_mass"
energy_strategy = "full_energy"

## plot set-up

In [ ]:
min_fontsize = 13
midle_fontsize = 17
max_fontsize = 21

# police font
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['font.monospace'] = 'Lucida Console'


## init for the project

In [ ]:
subject = "pc" # pc stand for physical (only the physical content (e.g., material, energy, surface) of the flows is analysed)
method = "dup" # What is the method used in this code to get the inventory of the technosphere flows, in this work, circularity is obtained by duplication of these flows to the biosphere matrix, here of subscript: "dup"

software_provider = "bw"
software_version = "25"

database_provider = "ecoinvent"   # "ecoinvent" or "bafu"

ecospold_folder = None # if the import is based on spold file

database_version = "3.12"
database_systemmodel = "consequential" # can be 'cutoff', 'apos', and 'consequential' (ecoinvent) or 'not-precised' (e.g., bafu)

# For project naming conventions
# In init flow treatment, we can choose to ignore the flows that have in their names: "water" (high influence on the total mass)
water_suffix = "no-w" if exclude_water else "all"

# mass strategy
if mass_strategy == "full_mass":
     mass_suffix = "fm" 
elif mass_strategy == "water_mass":
     mass_suffix = "wm"
elif mass_strategy == "dry_mass":
     mass_suffix = "dm"
# energy strategy   
if energy_strategy == "full_energy":
     energy_suffix = "fe" 
elif energy_strategy == "renewable_energy":
     energy_suffix = "re"
elif energy_strategy == "non-renewable_energy":
     energy_suffix = "nre"

project_name = f"{subject}_{method}_{software_provider}-{software_version}_{database_provider}-{database_version}-{database_systemmodel}_{mass_suffix}-{energy_suffix}"
# With water suffix: project_name = f"{subject}_{method}_{software_provider}-{software_version}_{database_provider}-{database_version}-{database_systemmodel}_{water_suffix}"

technosphere_db_name = f"{database_provider}-{database_version}-{database_systemmodel}"
biosphere_db_name = f"{database_provider}-{database_version}-biosphere" if database_provider == "ecoinvent" else "biosphere3"

def setup_project(): #also in functions.py file
        """Set up a new Brightway2 project and import ecoinvent if missing"""
        if project_name not in bd.projects:
            bd.projects.create_project(project_name)
        bd.projects.set_current(project_name)
        print(f"The project '{project_name}' is set as current.")
        # the principles are applied to the most used LCA database, ecoinvent [https://ecochain.com/blog/lci-databases-in-lca/]
        # the cutoff system model is the simplest to understand [https://support.ecoinvent.org/system-models-1]
        if technosphere_db_name not in bd.databases:
            if database_provider == "ecoinvent":
                print(f"Importing {technosphere_db_name} database...")
                bi.import_ecoinvent_release(
                    database_version,
                    database_systemmodel,
                    ei_username,
                    ei_password
                )
        print("Setup" \
        " complete.")
        setup_complete = True

setup_project()    

In [ ]:
# bd.projects.set_current("another_project")
# bd.projects.delete_project('pc_dup_bw-25_ecoinvent-3.12-consequential_fm-fe', delete_dir=True)

### test

In [ ]:
db = bd.Database(technosphere_db_name)
activities = list(db) # type: ignore
print(f"Total activities in DB: {len(activities)}")

# Inspect exchange types on a few activities
for act in activities[:5]:
    exchanges = list(act.exchanges())
    print(f"\n--- {act.get('name')} ---")
    for exc in exchanges:
        print(f"  type={exc.get('type')}, name={exc.get('name')}, amount={exc.get('amount')}")


# 🎬 exe

## exe analysis

### count cut-off product-flows

In [ ]:
# With each module taken separately:
# --- 1. Import BurdenFreeAnalyzer --- 

bfa = BurdenFreeAnalyzer(
    project_name=project_name,
    database_provider=database_provider,
    ecospold_folder=ecospold_folder,
    technosphere_db_name=technosphere_db_name,
    database_version=database_version,
    database_systemmodel=database_systemmodel
)

# --- 2. Analyze Burden-Free Activities --- 

# Delete cache to force fresh analysis
# cache_file = f"results/json/{project_name}_added_product_flows.json"
# if os.path.exists(cache_file):
#     os.remove(cache_file)
#     print(f"🗑️ Deleted cache: {cache_file}")

burden_free_activities = bfa.analyze_all_databases()

# --- 3. Initialize Biosphere Flow Manager ---
biosphere_flow_manager = BiosphereFlowManager(biosphere_db_name)  # Your biosphere DB

# --- 4. Count activities in your specific JSON file --- 
counts = bfa.count_activities_in_json(
    filename="pc_dup_bw-25_ecoinvent-3.12-consequential_fm-fe_added_product_flows.json"
)

# # With all modules combined:
# cut_off_uprs = clci.burdenfreeanalyzer(
#     project_name=project_name,
#     database_provider=database_provider,
#     ecospold_folder=ecospold_folder,
#     technosphere_db_name=technosphere_db_name,
#     database_version=database_version,
#     database_systemmodel=database_systemmodel
# )

clci.biosphereflowmanager(biosphere_db_name)  # Your biosphere DB
biosphere_flow_manager.process(burden_free_activities)

# counts = cut_off_uprs.count_activities_in_json(filename="pc_dup_bw-25_ecoinvent-3.11-cutoff_fm-fe_added_product_flows.json")

# readable output:
for db_name, activity_count in counts.items():
    print(f"Database '{db_name}': {activity_count} activities")

### ⚖️ analyse the causes of energy imbalances

In [ ]:
from bw2data import Database

def analyze_electricity_fossil_activities(project_name=None):
    """
    Analyze electricity production activities from fossil resources and their elementary flows.
    """
    if project_name:
        bd.projects.set_current(project_name)

    fossil_keywords = ['natural gas', 'oil', 'coal', 'nuclear'] 
    exclude_keywords = ['aluminium industry']

    # Find all matching activities
    matching_activities = []
    for db_name in bd.databases:
        db = Database(db_name)
        for act in db: # type: ignore
            act_name = act.get('name', '').lower()
            # Must start with electricity production AND contain fossil keyword
            # AND NOT contain any exclusion keywords
            if (act_name.startswith('electricity production') and
                any(fossil in act_name for fossil in fossil_keywords) and
                not any(excl in act_name for excl in exclude_keywords)):
                matching_activities.append(act)

    total_activities = len(matching_activities)

    # Initialize collections
    all_co2_flows = {}  # {compartment: set(flow_names)}
    all_heat_flows = {}  # {compartment: set(flow_names)}
    activities_with_co2 = 0
    activities_with_heat = 0
    first_three_examples = []

    # Analyze each activity
    for i, act in enumerate(matching_activities):
        act_key = (act['database'], act['code'])
        act_name = act.get('name', '')

        try:
            data = bd.get_activity(act_key)
            if data is None:
                continue

            act_co2_flows = {}
            act_heat_flows = {}

            for exc in data.exchanges():
                if exc['type'] != 'biosphere':
                    continue

                flow = bd.get_activity(exc['input'])
                if flow is None:
                    continue

                flow_name = flow.get('name', '')
                flow_name_lower = flow_name.lower()
                compartment = flow.get('categories', [None])[0] or 'unknown'

                # Check for carbon AND dioxide (case insensitive)
                if 'carbon' in flow_name_lower and 'dioxide' in flow_name_lower:
                    if compartment not in act_co2_flows:
                        act_co2_flows[compartment] = set()
                    act_co2_flows[compartment].add((flow_name, exc['amount']))
                    if compartment not in all_co2_flows:
                        all_co2_flows[compartment] = set()
                    all_co2_flows[compartment].add(flow_name)

                # Check for heat AND (waste OR losses)
                if 'heat' in flow_name_lower and ('waste' in flow_name_lower or 'losses' in flow_name_lower):
                    if compartment not in act_heat_flows:
                        act_heat_flows[compartment] = set()
                    act_heat_flows[compartment].add((flow_name, exc['amount']))
                    if compartment not in all_heat_flows:
                        all_heat_flows[compartment] = set()
                    all_heat_flows[compartment].add(flow_name)

            # Count activities with at least one matching flow
            if sum(len(v) for v in act_co2_flows.values()) > 0:
                activities_with_co2 += 1
            if sum(len(v) for v in act_heat_flows.values()) > 0:
                activities_with_heat += 1

            # Store first 3 examples with details
            if i < 3 and (act_co2_flows or act_heat_flows):
                example = {
                    'process': act_name,
                    'co2_flows': [],
                    'heat_flows': []
                }
                for comp, flows in act_co2_flows.items():
                    for flow_name, amount in flows:
                        example['co2_flows'].append({
                            'flow_name': flow_name,
                            'amount': amount,
                            'compartment': comp
                        })
                for comp, flows in act_heat_flows.items():
                    for flow_name, amount in flows:
                        example['heat_flows'].append({
                            'flow_name': flow_name,
                            'amount': amount,
                            'compartment': comp
                        })
                first_three_examples.append(example)

        except Exception as e:
            print(f"Error processing {act_key}: {e}")
            continue

    # Calculate ratios
    ratio_co2 = activities_with_co2 / total_activities if total_activities > 0 else 0
    ratio_heat = activities_with_heat / total_activities if total_activities > 0 else 0

    # Format CO2 flows by compartment
    co2_by_compartment = {comp: sorted(list(flows)) for comp, flows in all_co2_flows.items()}
    heat_by_compartment = {comp: sorted(list(flows)) for comp, flows in all_heat_flows.items()}

    # Create results DataFrame
    results_df = pd.DataFrame({
        'number of process in the LCA project': [total_activities],
        'ratio for CO2': [ratio_co2],
        'ratio for fatal heat': [ratio_heat],
        'example of upr with CO2 recorded': [', '.join([f for flows in all_co2_flows.values() for f in list(flows)[:3]]) if all_co2_flows else 'None'],
        'example of UPr with heat losses recorded': [', '.join([f for flows in all_heat_flows.values() for f in list(flows)[:3]]) if all_heat_flows else 'None'],
        'CO2 flows by compartment': [co2_by_compartment],
        'heat flows by compartment': [heat_by_compartment]
    })

    return results_df, first_three_examples

# Execute and print the final results
print("Analyzing electricity production activities from fossil resources...")
print(project_name)
energy_df = analyze_electricity_fossil_activities(project_name=project_name)
energy_df

In [ ]:
def get_processes_without_co2(project_name=None):
    """
    Get the names of electricity production processes from fossil resources that have NO CO2 emissions.
    Explicitly excludes nuclear and renewable sources.
    """
    if project_name:
        bd.projects.set_current(project_name)

    fossil_keywords = ['natural gas', 'oil', 'coal'] # The goal is to find the electricity production activities that record the emission of CO2 but not the fatal heat
    exclude_keywords = ['nuclear', 'aluminium industry']

    # Find all matching activities
    matching_activities = []
    for db_name in bd.databases:
        db = Database(db_name)
        for act in db: # type: ignore
            act_name = act.get('name', '').lower()
            # Must start with electricity production AND contain fossil keyword
            # AND NOT contain any exclusion keywords
            if (act_name.startswith('electricity production') and
                any(fossil in act_name for fossil in fossil_keywords) and
                not any(excl in act_name for excl in exclude_keywords)):
                matching_activities.append(act)

    no_co2_processes = []

    # Check each activity for CO2 flows
    for act in matching_activities:
        act_key = (act['database'], act['code'])
        act_name = act.get('name', '')

        try:
            data = bd.get_activity(act_key)
            if data is None:
                continue

            has_co2 = False
            for exc in data.exchanges():
                if exc['type'] != 'biosphere':
                    continue

                flow = bd.get_activity(exc['input'])
                if flow is None:
                    continue

                flow_name_lower = flow.get('name', '').lower()
                if 'carbon' in flow_name_lower and 'dioxide' in flow_name_lower:
                    has_co2 = True
                    break

            if not has_co2:
                no_co2_processes.append(act_name)

        except Exception as e:
            print(f"Error processing {act_key}: {e}")
            continue

    return no_co2_processes
# Example usage:
no_co2_processes = get_processes_without_co2(project_name=project_name)
print(f"\nProcesses with NO CO2 emissions ({len(no_co2_processes)}):")
for proc in no_co2_processes:
    print(f"  - {proc}")

### analyse an auxiliary process's circularity

📖 auxiliary processes such as in (Amatuni et al. (2025) [https://doi.org/10.1111/jiec.13538])

In [ ]:
def find_processes_by_name(name_part, loc):
    """Find all processes matching a name pattern"""
    matches = []
    for db_name in bd.databases:
        db = bd.Database(db_name)
        for act in db: # type: ignore
            if name_part.lower() in act.get('name', '').lower() and loc.lower() in act.get('location', '').lower():
                matches.append(act)
    return matches
    matches.append(act)


# Find all anaerobic digestion processes
process_name = "market for fibre, jute"
location = "GLO"
target_processes = find_processes_by_name(process_name, location)

if not target_processes:
    raise ValueError(f"No processes found matching '{process_name}'")

print(f"Found {len(target_processes)} matching processes:")
for i, proc in enumerate(target_processes):
    print(f"{i+1}. {proc.get('name')} ({proc.get('location')}) - Key: {proc.key}")

# Select the first one (or choose specific one)
target_process = target_processes[0]  # Change index to select different version
print(f"\nSelected process: {target_process.get('name')}")
print(f"Location: {target_process.get('location')}")
print(f"Key: {target_process.key}")
print(f"Database: {target_process['database']}")

In [ ]:
# Initialize calculator
calculator = CircularityCalculator(
    project_name=project_name,
    excluded_flows=excluded_flows,
    valuable_water_compartments=valuable_water_compartments,
    exclude_water=exclude_water,
    technosphere_db_name=technosphere_db_name,
    biosphere_db_name=biosphere_db_name,
    mass_strategy=mass_strategy,
    energy_strategy=energy_strategy
)

# Compute circularity directly with functional unit (no setup needed!)
try:
    fu = {target_process.key: 1}
    result_df, inefficiency, efficiency, detailed_flows = calculator.compute_circularity_efficiency_variables(
        functional_unit=fu,
        save_csv=True
    )
    
    print("\n" + "="*50)
    print("CIRCULARITY RESULTS")
    print("="*50)
    
    if result_df is not None:
        display(result_df)
    else:
        print("No results returned")
        
except Exception as e:
    print(f"Error computing circularity: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
print("\n" + "="*50)
print("TOP 5 FLOWS BY MASS (kg-eq)")
print("="*50)
top_mass = detailed_flows.nlargest(5, 'Mass_Equivalent_kg') if detailed_flows is not None else pd.DataFrame()
for _, row in top_mass.iterrows():
    print(f"   {row['Flow Name']}: {row['Mass_Equivalent_kg']:.2f} kg")

print("\n" + "="*50)
print("TOP 5 FLOWS BY ENERGY (MJ-eq)")
print("="*50)
top_energy = detailed_flows.nlargest(5, 'Energy_Equivalent_MJ') if detailed_flows is not None else pd.DataFrame()
for _, row in top_energy.iterrows():
    print(f"   {row['Flow Name']}: {row['Energy_Equivalent_MJ']:.2f} MJ")

# print("\n" + "="*50)
# print("DETAILED FLOWS (all)")
# print("="*50)
# display(detailed_flows)

# if detailed_flows is not None and not detailed_flows.empty:
#     output_dir = Path("results/csv")
#     output_dir.mkdir(parents=True, exist_ok=True)
#     detailed_path = output_dir / f"{project_name}_detailed_flows.csv"
#     detailed_flows.to_csv(detailed_path, index=False)
#     print(f"\nDetailed flows saved to: {detailed_path}")
# else:
#     print("\nNo detailed flows available to save.")

### LCIA test 
- for activity browser visualisation
- for easier contribution visualisation (e.g., with brightway)

In [ ]:
from circularity_lci.lcia import create_circularity_lcia_methods

# Add progress tracking
lcia_builder = create_circularity_lcia_methods(
    project_name=project_name,
    excluded_flows=excluded_flows,
    valuable_water_compartments=valuable_water_compartments,
    exclude_water=exclude_water,
    technosphere_db_name=technosphere_db_name,
    biosphere_db_name=biosphere_db_name,
    mass_strategy=mass_strategy,
    energy_strategy=energy_strategy
)

# Create all 44 methods
all_methods = lcia_builder.create_all_methods()
# Create 32 mass methods
# mass_lcia = lcia_builder.create_all_mass_methods(include_water=True)
# # Create 12 energy methods
# energy_lcia = lcia_builder.create_all_energy_methods()

In [ ]:
fu = {target_process.key: 1}
list(bd.methods)[-12]
lca = bc.LCA(fu, ('Circularity Indicator',
 'Energy Flows',
 'Cumulative Natural Resources Inputs (fe)'))
lca.lci()
lca.lcia()
lcia_result = lca.score
print(lca.score) 
# for waste E: 7.796615*10^5 but obtained 776174.6689399256
# for V_E: 11.691330e+07	but obtained 16913303.76174263

#### consequential results

In [ ]:
# Get the inventory used by the calculator
flows_df = calculator.get_inventory_flows(functional_unit=fu)

# Identify natural resource flows
resources = flows_df[
    flows_df["Type"].str.lower().str.contains("resource", na=False)
].copy()

print("Number of resource flows:", len(resources))
print("Negative amounts:", (resources["Amount"] < 0).sum())
print("Positive amounts:", (resources["Amount"] > 0).sum())

display(resources[resources["Amount"] < 0])

## exe calc setup

In [ ]:
def main_circularity():
    """
    Function to handle the initial steps of circularity calculation.
    """
    project_name = bd.projects.current
    tracker = ProgressTracker(project_name=project_name)
    burden_free_analyzer = BurdenFreeAnalyzer(
        project_name=project_name,
        database_provider=database_provider,
        ecospold_folder=ecospold_folder,        
        technosphere_db_name=technosphere_db_name,
        database_version=database_version,
        database_systemmodel=database_systemmodel
    )
    
    bf_manager = BiosphereFlowManager(biosphere_db_name=biosphere_db_name)

    # Step 1: Analyze burden-free activities
    if not tracker.is_step_completed("analysis"):
        start = time.time()
        burden_free = burden_free_analyzer.analyze_all_databases()
        tracker.update_step("analysis", "completed",
                            data={k: len(v) for k, v in burden_free.items()},
                            execution_time=f"{time.time() - start:.2f}s")
    else:
        print("Step 1 already completed. Reloading data from project...")
        burden_free = burden_free_analyzer.analyze_all_databases()

    # Step 2: Create biosphere flows
    if not tracker.is_step_completed("biosphere_flows"):
        start = time.time()
        bf_manager.process(burden_free)
        tracker.update_step("biosphere_flows", "completed",
                            data="Flows and exchanges created",
                            execution_time=f"{time.time() - start:.2f}s")
    else:
        print("Step 2 already completed.")

    # Create a new setup for circularity calculation
    calculator = MultiLCACalculator(technosphere_db_name=technosphere_db_name)
    calculator.precompute_locations()

    # Get user input for the calculation setup name
    setup_name = input("Enter a name for the circularity calculation setup: ").strip()
    
    # If setup already exists, you might want to use it or overwrite
    if setup_name in bd.calculation_setups:
        overwrite = input(f"Setup '{setup_name}' already exists. Overwrite? (y/n): ").strip().lower()
        if overwrite != 'y':
            setup_name = input("Enter a new name: ").strip()

    try:
        # Select activities
        functional_units = calculator.select_activities()

        # Flatten the methods list
        methods = list(bd.methods)
        if methods:
            # Randomly select a method (or you could let user choose)
            random_method = methods[np.random.randint(len(methods))]
            print(f"Randomly selected method: {random_method}")

            # Create calculation setup
            bd.calculation_setups[setup_name] = {
                "inv": [dict([fu]) for fu in functional_units],
                "ia": [random_method],
                "description": f"Circularity calculation with {len(functional_units)} functional units"
            }

            print(f"\nCalculation setup '{setup_name}' created successfully!")

            # Initialize CircularityCalculator once
            circularitycalculator = CircularityCalculator(
                project_name, 
                excluded_flows, 
                valuable_water_compartments, 
                exclude_water, 
                technosphere_db_name, 
                biosphere_db_name, 
                mass_strategy, 
                energy_strategy
            )
            
            # Get inventory flows
            flows_df = circularitycalculator.get_inventory_flows(setup_name)
            
            if flows_df.empty:
                print("Warning: No inventory flows found!")
                return

            # Compute efficiency circularity variables
            efficiency_df, LFI, CFI, detailed_flows_df = circularitycalculator.compute_circularity_efficiency_variables(
                flows_df,
                setup_name=setup_name
            )

            print("\n" + "="*50)
            print("EFFICIENCY CIRCULARITY RESULTS")
            print("="*50)
            print(efficiency_df)
            print(f"\nInefficiency (η-): {LFI}")
            print(f"Efficiency (η+): {CFI}")

            # Compute EMF circularity indicators
            EMF_df, standard_results, detailed_flows = circularitycalculator.compute_circularity_EMF_indicators(
                flows_df, setup_name=setup_name)

            print("\n" + "="*50)
            print("EMF CIRCULARITY RESULTS")
            print("="*50)
            print(EMF_df)

            # Compute ISO 59020 recycled input rate
            ISO59020_df, standard_results, detailed_flows = circularitycalculator.compute_iso59020_recycled_input_rate(
                flows_df, setup_name=setup_name)

            print("\n" + "="*50)
            print("ISO 59020 RECYCLED INPUT RATE")
            print("="*50)
            print(ISO59020_df)

            # Compute ISO 59020 recycled output rate
            ISO59020_output_df, standard_results, detailed_flows = circularitycalculator.compute_iso59020_recycled_output_rate(
                flows_df, setup_name=setup_name)

            print("\n" + "="*50)
            print("ISO 59020 RECYCLED OUTPUT RATE")
            print("="*50)
            print(ISO59020_output_df)

            # Save all results
            output_dir = os.path.join("results", "csv", f"{project_name}")
            if not os.path.exists(output_dir):
                os.makedirs(output_dir)
            
            # Determine water suffix for file names
            water_suffix = "exclude_water" if exclude_water else "include_water"
            
            # Save results
            results_path = os.path.join(output_dir, f"{setup_name}_{water_suffix}_eta_circularity_results.csv")
            detailed_path = os.path.join(output_dir, f"{setup_name}_{water_suffix}_detailed_flows.csv")
            EMF_path = os.path.join(output_dir, f"{setup_name}_{water_suffix}_EMF_circularity_results.csv")
            ISO59020_input_path = os.path.join(output_dir, f"{setup_name}_{water_suffix}_ISO59020_input_rate.csv")
            ISO59020_output_path = os.path.join(output_dir, f"{setup_name}_{water_suffix}_ISO59020_output_rate.csv")

            # Save DataFrames (handle None cases)
            if efficiency_df is not None:
                efficiency_df.to_csv(results_path, index=True)
            if detailed_flows_df is not None:
                detailed_flows_df.to_csv(detailed_path, index=True)
            if EMF_df is not None:
                EMF_df.to_csv(EMF_path, index=True)
            if ISO59020_df is not None:
                ISO59020_df.to_csv(ISO59020_input_path, index=True)
            if ISO59020_output_df is not None:
                ISO59020_output_df.to_csv(ISO59020_output_path, index=True)

            print(f"\n✅ Results saved to folder: {output_dir}")
            print(f"   - {os.path.basename(results_path)} (η- / η+)")
            print(f"   - {os.path.basename(EMF_path)} (EMF indicators)")
            print(f"   - {os.path.basename(detailed_path)} (detailed flows)")
            print(f"   - {os.path.basename(ISO59020_input_path)} (ISO 59020 input rate)")
            print(f"   - {os.path.basename(ISO59020_output_path)} (ISO 59020 output rate)")

            # Optional: Dry mass diagnostic
            if detailed_flows_df is not None and not detailed_flows_df.empty:
                dry_mass_comparison_df = circularitycalculator.compare_dry_mass_factors(
                    detailed_flows_df, tolerance=1e-6
                )
                if not dry_mass_comparison_df.empty:
                    dry_mass_path = os.path.join(
                        output_dir,
                        f"{setup_name}_{water_suffix}_dry_mass_comparison.csv"
                    )
                    dry_mass_comparison_df.to_csv(dry_mass_path, index=True)
                    print(f"   - {os.path.basename(dry_mass_path)} (dry mass comparison)")

        else:
            print("No methods found in the database.")

    except Exception as e:
        print(f"Error: {str(e)}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main_circularity()

## exe full/sample db

### get the intrinsic physical circularity results

In [ ]:
# Create an instance of CircularityDatabaseAnalyzer
analyzer = CircularityDatabaseAnalyzer(
    project_name = bd.projects.current,
    isic_section_map=isic_section_map,
    isic_division_map=isic_division_map,
    location_to_un_group=location_to_un_group,
    excluded_flows=excluded_flows,
    valuable_water_compartments=valuable_water_compartments,
    exclude_water=exclude_water,
    technosphere_db_name=technosphere_db_name,
    biosphere_db_name=biosphere_db_name,
    mass_strategy=mass_strategy,
    energy_strategy=energy_strategy
)

unitprocesses_number = 1000
# Compute circularity for all processes
results_df = analyzer.compute_circularity_for_all_processes_sequential(sample_size=unitprocesses_number)

### histogram per circularity variables

In [ ]:

def plot_circularity_distribution(loaded_df, circularity_variable='LFI_kg', group_by='sections', bins=None, isic_section_map=None):
    """
    Plot distribution of circularity values grouped by categories.

    Args:
        loaded_df (pd.DataFrame): DataFrame containing circularity results.
        circularity_variable (str): The circularity variable to plot (e.g., 'eta-_kg', 'eta+_kg', 'CFI_kg', 'LFI_kg').
        group_by (str): The grouping variable for the colors ('location' or 'ISIC Section').
        bins (list): Bin edges for the histogram. If None, uses default bins.
        isic_section_map (dict): Dictionary mapping ISIC section codes to full names.
    """
    if circularity_variable not in loaded_df.columns:
        raise ValueError(f"Column '{circularity_variable}' not found in loaded_df.")

    if group_by not in ['location', 'isic_section', 'un_group']:
        raise ValueError("group_by must be 'location' or 'ISIC_Section' or 'un_group'.")

    if group_by == 'un_group':
        group_col = 'UN Group'
        title_suffix = 'by UN Group'
    elif group_by == 'location':
        group_col = 'LOCATION Code'
        title_suffix = 'by Location'
    else:
        group_col = 'ISIC Section'
        title_suffix = 'by ISIC Section'

    # Define bins if not provided
    if bins is None:
        bins = [0, 0.2, 0.4, 0.6, 0.8, 1.0, float('inf')]
    
    bin_labels = ['0-0.2', '0.2-0.4', '0.4-0.6', '0.6-0.8', '0.8-1.0', '>1.0']
    
    # Create binned data
    data = loaded_df.copy()
    data['bin'] = pd.cut(data[circularity_variable], bins=bins, labels=bin_labels, include_lowest=True)
    
    # Create cross-tabulation for stacked bars
    cross_tab = pd.crosstab(data['bin'], data[group_col])
    
    # Calculate percentages for annotations
    cross_tab_percent = cross_tab.div(cross_tab.sum(axis=1), axis=0) * 100
    
    # Get number of unique groups for color selection
    unique_groups = cross_tab.columns
    n_groups = len(unique_groups)
    
    # IMPROVED COLOR SELECTION - Choose based on number of groups
    if n_groups <= 8:
        # For few groups, use highly distinct colors
        colors = sns.color_palette("Set2", n_groups)
    elif n_groups <= 12:
        # For medium number of groups, use tab20
        colors = sns.color_palette("tab20", n_groups)
    elif n_groups <= 20:
        # For more groups, use color cube for maximum distinction
        colors = sns.color_palette("husl", n_groups)
    else:
        # For many groups, use the most distinct palette available
        colors = sns.color_palette("gist_ncar", n_groups)
    
    # Create the plot
    plt.figure(figsize=(15, 10)) # Slightly wider for better legend spacing
    
    # Create stacked bar plot with improved colors
    ax = cross_tab.plot(kind='bar', stacked=True, figsize=(15, 10),color=colors)
    
    # ADD PERCENTAGE ANNOTATIONS
    # Iterate through each bar (bin) and each segment within the bar
    for i, (bin_label, row) in enumerate(cross_tab.iterrows()):
        # Calculate cumulative height for positioning
        cumulative_height = 0
        total_in_bin = row.sum()
        
        for j, group in enumerate(unique_groups):
            count = row[group]
            percentage = cross_tab_percent.loc[bin_label, group] # type: ignore
            
            # Only annotate if the segment is large enough to be visible
            # Only annotate if the segment is large enough to be visible
            min_percentage = 5  # Minimum percentage threshold
            min_count = 100      # Minimum absolute count threshold
            min_height_ratio = 0.03  # Minimum height ratio of the segment (3% of total bar height)
            # if percentage >= 5:  # Only show percentages for segments >= 5%
            if (percentage >= min_percentage and 
                count >= min_count and 
                (count / total_in_bin) >= min_height_ratio):
                # Position the text in the middle of the segment
                y_pos = cumulative_height + count / 2
                
                # Add text annotation
                ax.text(
                    i,  # x position (bar index)
                    y_pos,  # y position (middle of segment)
                    f'{percentage:.0f}%',  # Text (rounded percentage)
                    ha='center', 
                    va='center',
                    fontsize=min_fontsize,
                    fontweight='bold',
                    color = 'black' # color='white' if percentage > 20 else 'black'  # White text on dark segments
                )
            
            cumulative_height += count
    
    plt.xlabel(f'{circularity_variable} Ranges')
    plt.ylabel('Count of Activities')
    plt.title(f'Distribution of {circularity_variable} {title_suffix}', fontsize=max_fontsize, fontweight='bold')
    plt.xticks(rotation=45)
    
    # CREATE IMPROVED LEGEND LABELS
    if group_by == 'isic_section' and isic_section_map is not None:
        # Create legend labels using the mapping
        legend_labels = []
        for group in unique_groups:
            if group in isic_section_map:
                legend_labels.append(f"{group}: {isic_section_map[group]}")
            else:
                legend_labels.append(f"{group}: Unknown")
        
        # Create custom legend
        handles, _ = ax.get_legend_handles_labels()
        plt.legend(
            handles=handles,
            labels=legend_labels,
            bbox_to_anchor=(0.5, 1.1),  # Position above the plot
            loc='lower center',          # Center the legend horizontally
            title=group_col,
            fontsize=min_fontsize,
            title_fontsize=midle_fontsize,
            ncol= 2 # len(unique_groups)     # Adjust number of columns if needed
        )

    elif group_by =='un_group' and location_to_un_group is not None:
        # Create legend labels using the mapping for UN groups
        legend_labels = []
        for group in unique_groups:
            if group in location_to_un_group:
                legend_labels.append(f"{group}: {location_to_un_group[group]}")
            else:
                legend_labels.append(f"{group}: Unknown")
        
        # Create custom legend
        handles, _ = ax.get_legend_handles_labels()
        plt.legend(
            handles=handles,
            labels=legend_labels,
            bbox_to_anchor=(0.5, 1.1),  # Position above the plot
            loc='lower center',          # Center the legend horizontally
            title=group_col,
            fontsize=min_fontsize,
            title_fontsize=midle_fontsize,
            ncol= 2 # len(unique_groups)     # Adjust number of columns if needed
        )
    else:
        # Standard legend for other group types
        plt.legend(
            bbox_to_anchor=(1.05, 1), 
            loc='best', 
            title=group_col,
            fontsize= min_fontsize,
            title_fontsize= midle_fontsize
        )
    
    # Add grid for better readability
    plt.grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    
    # save the graphs
    save_path = f"results/histogram/1-plot-per-CI/{project_name}_{unitprocesses_number}uprs"
    os.makedirs(save_path, exist_ok=True)
    filename = f"{save_path}/{circularity_variable}_{group_by}.png"
    plt.savefig(filename, dpi=1000, bbox_inches='tight')
    print(f"Plot saved as: {filename}")
    
    plt.show()
    
    # Print some statistics
    print(f"Total activities: {len(loaded_df)}")
    print(f"Number of {group_col} groups: {n_groups}")
    print(f"\nDistribution of {circularity_variable}:")
    for bin_label in bin_labels:
        count = len(data[data['bin'] == bin_label])
        percentage = (count / len(data)) * 100
        print(f"{bin_label}: {count} activities ({percentage:.1f}%)")
    
    # Print detailed percentage breakdown by group for each bin
    print(f"\nDetailed percentage breakdown by {group_col}:")
    for bin_label in bin_labels:
        if bin_label in cross_tab_percent.index:
            print(f"\n{bin_label} range:")
            bin_percentages = cross_tab_percent.loc[bin_label]
            for group in unique_groups:
                percentage = bin_percentages[group]
                if percentage > 0:  # type: ignore # Only show groups with presence in this bin
                    print(f"  {group}: {percentage:.1f}%")

# Example usage
if __name__ == "__main__":
    # Assuming loaded_df is your DataFrame with circularity results
    # Plot with ISIC sections using the mapping
    plot_circularity_distribution(
        loaded_df, # type: ignore load the pkl file 
        circularity_variable='LFI_kg', 
        group_by='isic_section', 
        isic_section_map=isic_section_map
    )
    plot_circularity_distribution(
        loaded_df, # type: ignore load the pkl file 
        circularity_variable='CFI_kg', 
        group_by='isic_section', 
        isic_section_map=isic_section_map
    )
    plot_circularity_distribution(
        loaded_df, # type: ignore load the pkl file 
        circularity_variable='LFI_MJ', 
        group_by='isic_section', 
        isic_section_map=isic_section_map
    )
    plot_circularity_distribution(
        loaded_df, # type: ignore load the pkl file 
        circularity_variable='CFI_MJ', 
        group_by='isic_section', 
        isic_section_map=isic_section_map
    )
    
    # Other plots (the mapping won't be used for these)
    plot_circularity_distribution(
        loaded_df, # type: ignore load the pkl file 
        circularity_variable='eta-_kg', 
        group_by='un_group'
    )
    plot_circularity_distribution(
        loaded_df, # type: ignore load the pkl file 
        circularity_variable='eta-_kg', 
        group_by='isic_section',
        isic_section_map=isic_section_map
    )
    plot_circularity_distribution(
        loaded_df, # type: ignore load the pkl file 
        circularity_variable='eta+_kg', 
        group_by='isic_section',
        isic_section_map=isic_section_map
    )
    plot_circularity_distribution(
        loaded_df, # type: ignore load the pkl file 
        circularity_variable='eta+_kg', 
        group_by='un_group'
    )
    plot_circularity_distribution(
        loaded_df, # type: ignore load the pkl file 
        circularity_variable='eta-_MJ', 
        group_by='un_group'
    )
    plot_circularity_distribution(
        loaded_df, # type: ignore load the pkl file 
        circularity_variable='eta-_MJ', 
        group_by='isic_section',
        isic_section_map=isic_section_map
    )
    plot_circularity_distribution(
        loaded_df, # type: ignore load the pkl file 
        circularity_variable='eta+_MJ', 
        group_by='un_group'
    )
    plot_circularity_distribution(
        loaded_df, # type: ignore load the pkl file 
        circularity_variable='eta+_MJ', 
        group_by='isic_section',
        isic_section_map=isic_section_map
    )

### live comparison energy-mass

In [ ]:
def plot_hybrid_scale_scatter(results_df, save_path_2="basic_plots", group_by="section", isic_section_map=None):
    """
    Create scatter plots of LFI, CFI, eta+, and eta- in kg vs MJ with a hybrid scale.
    """
    
    if results_df.empty:
        print("No data to plot.")
        return
    
    if group_by.lower() == "section":
        col = "ISIC Section"
    elif group_by.lower() == "division":
        col = "ISIC Division"
    elif group_by.lower() == "description":
        col = "ISIC Description"
    elif group_by.lower() == "location":
        col = "LOCATION Code"
    elif group_by.lower() == "un_group":
        col = "UN Group"
    else:
        raise ValueError("group_by must be 'section', 'division', 'un_group', or 'description'")
    
    os.makedirs(save_path_2, exist_ok=True)
    
    # Get unique groups and sort them for consistent color assignment
    unique_groups = sorted(results_df[col].dropna().unique())
    n_groups = len(unique_groups)
    
    # USE THE SAME COLOR LOGIC AS plot_circularity_distribution
    if n_groups <= 8:
        colors = sns.color_palette("Set2", n_groups)
    elif n_groups <= 12:
        colors = sns.color_palette("tab20", n_groups)
    elif n_groups <= 20:
        colors = sns.color_palette("husl", n_groups)
    else:
        colors = sns.color_palette("gist_ncar", n_groups)
        
    color_map = dict(zip(unique_groups, colors))
    
    # CREATE LEGEND LABELS
    if col == "ISIC Section" and isic_section_map is not None:
        legend_labels = []
        for group in unique_groups:
            if group in isic_section_map:
                legend_labels.append(f"{group}: {isic_section_map[group]}")
            else:
                legend_labels.append(f"{group}: Unknown")
    else:
        legend_labels = None

    # Define the transform classes
    class LinearLogTransform:
        input_dims = output_dims = 1
        def transform(self, a):
            a = np.clip(a, 1e-10, None)
            return np.where(a <= 1, a, np.log10(a) + 1)
        def inverted(self):
            return InvertedLinearLogTransform()

    class InvertedLinearLogTransform:
        input_dims = output_dims = 1
        def transform(self, a):
            return np.where(a <= 1, a, 10**(a - 1))
        def inverted(self):
            return LinearLogTransform()

    linear_log_transform = LinearLogTransform()
    inverted_linear_log_transform = InvertedLinearLogTransform()

    # Generic make_on_add function that works for any plot
    def make_on_add(current_local_artists, current_local_subsets, x_var, y_var, x_label, y_label):
        def on_add(sel):
            try:
                artist_id = current_local_artists.index(sel.artist)
                subset = current_local_subsets[artist_id]
                idx = sel.index
                
                row = subset.iloc[idx]
                sel.annotation.set(
                    text=(
                        f"Process: {row['Process Name']}\n"
                        f"Location: {row['Location']}\n"
                        f"Reference Product: {row['Reference Product']}\n"
                        f"Group: {row[col]}\n"
                        f"{x_label}: {row[x_var]:.4f}\n"
                        f"{y_label}: {row[y_var]:.4f}" # \n"
                        # f"CFI_kg: {row['CFI_kg']:.4f}\n"
                        # f"CFI_MJ: {row['CFI_MJ']:.4f}\n"
                        # f"η⁺_kg: {row['eta+_kg']:.4f}\n"
                        # f"η⁺_MJ: {row['eta+_MJ']:.4f}\n"
                        # f"η⁻_kg: {row['eta-_kg']:.4f}\n"
                        # f"η⁻_MJ: {row['eta-_MJ']:.4f}"
                    ),
                    fontsize=midle_fontsize
                )
            except (ValueError, IndexError):
                sel.annotation.set(visible=False)
        return on_add

    # Helper function to create scatter plots with hover
    def create_scatter_plot_with_hover(ax, x_col, y_col, title, x_label, y_label, x_ticks, y_ticks, save_filename):
        scatter_artists = []
        scatter_subsets = []
        
        for grp in unique_groups:
            subset = results_df[results_df[col] == grp]
            
            points = ax.scatter(
                subset[x_col].values,
                subset[y_col].values,
                color=color_map[grp],
                label=grp,
                alpha=0.4,
                s=10
            )
            
            scatter_artists.append(points)
            scatter_subsets.append(subset.reset_index(drop=True))
        
        ax.xaxis.set_major_locator(ticker.FixedLocator(x_ticks))
        ax.yaxis.set_major_locator(ticker.FixedLocator(y_ticks))
        ax.plot([min(x_ticks), max(x_ticks)], [min(y_ticks), max(y_ticks)], 'r--', alpha=1, label='y = x (Equality Line)')
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.set_title(title)
        ax.grid(True, alpha=0.3)
        
        if legend_labels is not None:
            handles, labels = ax.get_legend_handles_labels()
            equality_line_handle = handles[-1]
            equality_line_label = labels[-1]
            handles = handles[:-1]
            labels = legend_labels
            handles.append(equality_line_handle)
            labels.append(equality_line_label)
            
            ax.legend(
                handles=handles,
                labels=labels,
                bbox_to_anchor=(1.05, 1),
                loc='upper left',
                title=col,
                fontsize=min_fontsize,
                title_fontsize=midle_fontsize
            )
        else:
            ax.legend(
                bbox_to_anchor=(1.05, 1),
                loc='upper left',
                title=col,
                fontsize=min_fontsize,
                title_fontsize=midle_fontsize
            )
        
        plt.tight_layout()
        plt.savefig(save_filename, format='svg', dpi=1000, bbox_inches='tight')    
        
        # Add hover functionality
        cursor = mplcursors.cursor(scatter_artists, hover=True)
        cursor.connect("add", make_on_add(scatter_artists, scatter_subsets, x_col, y_col, x_label, y_label))
        
        return cursor

    # LFI scatter plot
    fig1, ax1 = plt.subplots(figsize=(15, 10))
    ax1.set_xscale('function', functions=(linear_log_transform.transform, inverted_linear_log_transform.transform))
    ax1.set_yscale('function', functions=(linear_log_transform.transform, inverted_linear_log_transform.transform))
    cursor1 = create_scatter_plot_with_hover(
        ax1, 'LFI_kg', 'LFI_MJ',
        f'Linear Flow Index: kg vs MJ (Hybrid Scale) by {col}',
        'LFI (kg-eq)', 'LFI (MJ-eq)',
        [0, 0.5, 1], [0, 0.5, 1], # [0, 0.5, 1], [0, 0.5, 1], # [0, 0.48, 0.49, 0.5, 0.51, 0.52, 1],
        f"{save_path_2}/LFI_hybrid_scale_by_{group_by}.svg"
    )
    plt.show()
    
    # CFI scatter plot
    fig2, ax2 = plt.subplots(figsize=(15, 10))
    ax2.set_xscale('function', functions=(linear_log_transform.transform, inverted_linear_log_transform.transform))
    ax2.set_yscale('function', functions=(linear_log_transform.transform, inverted_linear_log_transform.transform))
    cursor2 = create_scatter_plot_with_hover(
        ax2, 'CFI_kg', 'CFI_MJ',
        f'Circular Flow Index: kg vs MJ (Hybrid Scale) by {col}',
        'CFI (kg-eq)', 'CFI (MJ-eq)',
        [0, 0.5, 1], [0, 0.5, 1],
        f"{save_path_2}/CFI_hybrid_scale_by_{group_by}.svg"
    )
    plt.show()
    plt.close(fig2)
    
    # eta- scatter plot
    fig3, ax3 = plt.subplots(figsize=(15, 10))
    ax3.set_xscale('function', functions=(linear_log_transform.transform, inverted_linear_log_transform.transform))
    ax3.set_yscale('function', functions=(linear_log_transform.transform, inverted_linear_log_transform.transform))
    cursor3 = create_scatter_plot_with_hover(
        ax3, 'eta-_kg', 'eta-_MJ',
        f'Efficiency Negative: kg vs MJ (Hybrid Scale) by {col}',
        'η- (kg-eq)', 'η- (MJ-eq)',
        [0, 0.5, 1], [0, 0.5, 1],
        f"{save_path_2}/eta-_hybrid_scale_by_{group_by}.svg"
    )
    plt.show()
    
    # eta+ scatter plot
    fig4, ax4 = plt.subplots(figsize=(15, 10))
    ax4.set_xscale('function', functions=(linear_log_transform.transform, inverted_linear_log_transform.transform))
    ax4.set_yscale('function', functions=(linear_log_transform.transform, inverted_linear_log_transform.transform))
    cursor4 = create_scatter_plot_with_hover(
        ax4, 'eta+_kg', 'eta+_MJ',
        f'Efficiency Positive: kg vs MJ (Hybrid Scale) by {col}',
        'η+ (kg-eq)', 'η+ (MJ-eq)',
        [0, 0.5, 1], [0, 0.5, 1],
        f"{save_path_2}/eta+_hybrid_scale_by_{group_by}.svg"
    )
    plt.show()
    plt.close(fig4)

In [ ]:
# Define the save path
save_path_2 = f"results/plots/CI-kg_CI-mj/{project_name}_{unitprocesses_number}uprs"

if __name__ == "__main__":
    plot_hybrid_scale_scatter(loaded_df, save_path_2=save_path_2, group_by="section", isic_section_map=isic_section_map) # type: ignore load the pkl file

### live comparison efficiency/inefficiency

In [ ]:
def plot_cross_indicator_scatter(results_df, save_path="basic_plots", group_by="section", isic_section_map=None):
    """
    Create hybrid-scale scatter plots comparing indicators (LFI vs CFI, eta+ vs eta-) 
    for both kg and MJ bases, grouped by ISIC or location.
    """

    if results_df.empty:
        print("No data to plot.")
        return

    # Map grouping column
    group_cols = {
        "section": "ISIC Section",
        "division": "ISIC Division", 
        "description": "ISIC Description",
        "location": "LOCATION Code",
        "un_group": "UN Group"
    }
    col = group_cols.get(group_by.lower())
    if not col:
        raise ValueError("group_by must be 'section', 'division', 'description', 'location', or 'un_group'")

    os.makedirs(save_path, exist_ok=True)
    
    # Get unique groups and sort them for consistent color assignment
    unique_groups = sorted(results_df[col].dropna().unique())
    n_groups = len(unique_groups)
    
    # USE THE SAME COLOR LOGIC AS plot_circularity_distribution
    if n_groups <= 8:
        # For few groups, use highly distinct colors
        colors = sns.color_palette("Set2", n_groups)
    elif n_groups <= 12:
        # For medium number of groups, use tab20
        colors = sns.color_palette("tab20", n_groups)
    elif n_groups <= 20:
        # For more groups, use color cube for maximum distinction
        colors = sns.color_palette("husl", n_groups)
    else:
        # For many groups, use the most distinct palette available
        colors = sns.color_palette("gist_ncar", n_groups)
        
    # Create color map with sorted groups
    color_map = dict(zip(unique_groups, colors))
    
    # CREATE LEGEND LABELS (same as in plot_circularity_distribution)
    if col == "ISIC Section" and isic_section_map is not None:
        legend_labels = []
        for group in unique_groups:
            if group in isic_section_map:
                legend_labels.append(f"{group}: {isic_section_map[group]}")
            else:
                legend_labels.append(f"{group}: Unknown")
    else:
        legend_labels = None

    # Hybrid scale transformation
    class LinearLogTransform:
        input_dims = output_dims = 1
        def transform(self, a):
            a = np.clip(a, 1e-10, None)
            return np.where(a <= 1, a, np.log10(a) + 1)
        def inverted(self):
            return InvertedLinearLogTransform()

    class InvertedLinearLogTransform:
        input_dims = output_dims = 1
        def transform(self, a):
            return np.where(a <= 1, a, 10**(a - 1))
        def inverted(self):
            return LinearLogTransform()

    linear_log_transform = LinearLogTransform()
    inverted_linear_log_transform = InvertedLinearLogTransform()

    # Define pairs to plot
    pairs = [
        ("CFI_kg", "LFI_kg", "Circular vs Linear Flow Index (kg-eq)", "CFI (kg-eq)", "LFI (kg-eq)"),
        ("CFI_MJ", "LFI_MJ", "Circular vs Linear Flow Index (MJ-eq)", "CFI (MJ-eq)", "LFI (MJ-eq)"),
        ("eta+_kg", "eta-_kg", "Efficiency Positive vs Negative (kg-eq)", "η+ (kg-eq)", "η- (kg-eq)"),
        ("eta+_MJ", "eta-_MJ", "Efficiency Positive vs Negative (MJ-eq)", "η+ (MJ-eq)", "η- (MJ-eq)")
    ]

    for x_var, y_var, title, x_label, y_label in pairs:
        fig, ax = plt.subplots(figsize=(15, 10))
        ax.set_xscale('function', functions=(linear_log_transform.transform, inverted_linear_log_transform.transform))
        ax.set_yscale('function', functions=(linear_log_transform.transform, inverted_linear_log_transform.transform))

        # Store scatter artists and subsets for hover functionality
        scatter_artists = []  # Store scatter artists
        scatter_subsets = []  # Store subset corresponding to each artist

        for grp in unique_groups:
            subset = results_df[results_df[col] == grp]

            points = ax.scatter(
                subset[x_var].values,
                subset[y_var].values,
                color=color_map[grp],
                label=grp,
                alpha=0.4,
                s=10
            )

            scatter_artists.append(points)
            scatter_subsets.append(subset.reset_index(drop=True))

        # Define plot limits and draw balanced line
        lim = [0, 1]
        ax.plot([0, 1], [1, 0], 'r--', alpha=1, label='x + y = 1 (balanced)')
        ax.xaxis.set_major_locator(ticker.FixedLocator(lim))
        ax.yaxis.set_major_locator(ticker.FixedLocator(lim))
        ax.set_xlabel(x_label)
        ax.set_ylabel(y_label)
        ax.set_title(f"{title} (Hybrid Scale) by {col}")
        ax.grid(True, alpha=0.3)
        
        # Apply consistent legend styling
        if legend_labels is not None:
            handles, labels = ax.get_legend_handles_labels()
            
            # Remove the balanced line from handles/labels and add it separately
            balanced_line_handle = handles[-1]
            balanced_line_label = labels[-1]
            handles = handles[:-1]
            labels = legend_labels  # Use our custom labels for the groups
            
            # Add balanced line back
            handles.append(balanced_line_handle)
            labels.append(balanced_line_label)
            
            ax.legend(
                handles=handles,
                labels=labels,
                bbox_to_anchor=(1.05, 1), 
                loc='upper left',
                title=col,
                fontsize=min_fontsize,
                title_fontsize=midle_fontsize
            )
        else:
            ax.legend(
                bbox_to_anchor=(1.05, 1), 
                loc='upper left',
                title=col,
                fontsize=min_fontsize,
                title_fontsize=midle_fontsize
            )
            
        fig.tight_layout()
        fig.savefig(f"{save_path}/{x_var}_vs_{y_var}_hybrid_by_{group_by}.svg", dpi=1000, bbox_inches='tight')
        
        # Add interactive hover functionality
        # 1. Create shallow copies to 'freeze' the data for THIS specific plot
        local_artists = list(scatter_artists)
        local_subsets = list(scatter_subsets)

        # 2. Initialize cursor for this specific figure
        cursor = mplcursors.cursor(local_artists, hover=True)
        
        # 3. Create the closure, passing in the LOCAL versions of the lists
        def make_on_add(current_local_artists, current_local_subsets, c_x_var, c_y_var, c_x_label, c_y_label):
            def on_add(sel):
                try:
                    # Search only the artists belonging to this specific plot
                    artist_id = current_local_artists.index(sel.artist)
                    subset = current_local_subsets[artist_id]
                    idx = sel.index
                
                    row = subset.iloc[idx]
                    sel.annotation.set(
                        text=(
                            f"Process: {row['Process Name']}\n"
                            f"Location: {row['Location']}\n"
                            f"Reference Product: {row['Reference Product']}\n"
                            f"Group: {row[col]}\n"
                            f"{c_x_label}: {row[c_x_var]:.4f}\n"
                            f"{c_y_label}: {row[c_y_var]:.4f}"
                        ),
                        fontsize=min_fontsize
                    )
                except (ValueError, IndexError):
                    # If we hover over something else (like the red line), just hide the box
                    sel.annotation.set(visible=False)
            return on_add
        
        # 4. Connect the callback using the local data
        cursor.connect("add", make_on_add(local_artists, local_subsets, x_var, y_var, x_label, y_label))
        
        plt.show()

        # Optional: close the figure if needed
        # plt.close(fig)

In [ ]:
# Define the save path
save_path_3 = f"results/plots/comparison_pos-neg/{project_name}_{unitprocesses_number}uprs"

if __name__ == "__main__":
    plot_cross_indicator_scatter(loaded_df, save_path=save_path_3, group_by="section", isic_section_map=isic_section_map)  # type: ignore load the pkl file

# 💾 long run, save the results and load them

## save

In [ ]:
def save_results_df(results_df):
    """
    Simple save function using pickle (preserves everything)
    """
    save_dir = os.path.join("results", "pkl")
    os.makedirs(save_dir, exist_ok=True)

    water_suffix = "no-water" if exclude_water else "with-water"

    filepath = os.path.join(
        save_dir,
        f"{project_name}_{unitprocesses_number}uprs.pkl"
    )
    
    with open(filepath, 'wb') as f:
        pickle.dump(results_df, f)
    print(f"✅ Saved to {filepath}")
    print(f"   Shape: {results_df.shape}")
    return filepath

# Save with pickle (this preserves everything exactly)
# name the results consistently:
# - with how you considered the material flows: did you include water flows? (if yes, => "_with_water")
# - with how many unit processes you analysed (if 1000 random activities are selected, => "_1k")
save_results_df(results_df) 


## load

In [ ]:
unitprocesses_number = 1000
def load_results_df(filename):
    """
    Load results from a pickle file in the results_pkl directory.
    The user specifies the filename, but the path is automatically set to results_pkl/.
    """
    save_dir = os.path.join("results", "pkl")
    filepath = os.path.join(save_dir, f"{filename}.pkl")

    if os.path.exists(filepath):
        with open(filepath, 'rb') as f:
            df = pickle.load(f)
        print(f"✅ Loaded from {filepath}")
        print(f"   Shape: {df.shape}")
        return df
    else:
        raise FileNotFoundError(f"File not found: {filepath}")

# Example usage:
# Load the results by specifying the filename (without the .pkl extension)
loaded_df = load_results_df(f"{project_name}_{unitprocesses_number}uprs")


## checking

In [ ]:
def simple_check(original_df, loaded_df):
    """
    Simple check if DataFrames are equal
    """
    print("🔍 Simple Check:")
    print(f"   Shapes: {original_df.shape} vs {loaded_df.shape} → {'✅' if original_df.shape == loaded_df.shape else '❌'}")
    print(f"   Columns match: {'✅' if list(original_df.columns) == list(loaded_df.columns) else '❌'}")
    print(f"   Null counts: {original_df.isnull().sum().sum()} vs {loaded_df.isnull().sum().sum()} → {'✅' if original_df.isnull().sum().sum() == loaded_df.isnull().sum().sum() else '❌'}")
    print(f"   Are equal: {'✅ YES' if original_df.equals(loaded_df) else '❌ NO'}")
    
    # Show which columns have different null counts
    if not original_df.equals(loaded_df):
        print("\n🔍 Differences found:")
        for col in original_df.columns:
            orig_nulls = original_df[col].isnull().sum()
            loaded_nulls = loaded_df[col].isnull().sum()
            if orig_nulls != loaded_nulls:
                print(f"   {col}: {orig_nulls} vs {loaded_nulls} nulls")
    
    return original_df.equals(loaded_df)

# Simple check
simple_check(results_df, loaded_df)